## Lecture 4: Parallel Computing 1 — Peer Review MP1, Amdahl's Law & Multiprocessing

### ****Exercises 1:** MC π Serial**

**Write and time `estimate_pi_serial()`**

****Write** `estimate_pi_serial(num_samples)` — **a plain Python loop:****

**1. Draw two uniform random numbers $x, y ∈ [0, 1)$**

**2. Count a *hit* when $x^2 + y^2 \le 1$**

**3. Return 4 × hits/num samples**

****Time it:** use `time.perf counter()` with 3 runs, take `statistics.median()`. Try `num_samples = 10_000_000`.**

In [1]:
# From slide 38, modified by me to use main function

import math, random, time, statistics

def estimate_pi_serial(num_samples):
    inside_circle = 0
    for _ in range(num_samples):
        x, y = random.random(), random.random()
        if x*x + y*y <= 1:
            inside_circle += 1
    return 4 * inside_circle / num_samples


def main():
    num_samples = 10_000_000
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        pi_estimate = estimate_pi_serial(num_samples)
        times.append(time.perf_counter() - t0)
    t_serial = statistics.median(times)
    print(f"pi estimate: {pi_estimate:.6f} (error: {abs(pi_estimate-math.pi):.6f})")
    print(f"Serial time: {t_serial:.3f}s")


if __name__ == '__main__':
    main()

pi estimate: 3.140714 (error: 0.000878)
Serial time: 1.085s


**Questions:**

- **How accurate is the estimate? Run several times — does it vary?**

In [2]:
for i in range(5):
    main()

pi estimate: 3.142347 (error: 0.000755)
Serial time: 0.989s
pi estimate: 3.141688 (error: 0.000095)
Serial time: 0.995s
pi estimate: 3.140670 (error: 0.000923)
Serial time: 1.011s
pi estimate: 3.142138 (error: 0.000545)
Serial time: 1.008s
pi estimate: 3.141880 (error: 0.000287)
Serial time: 1.032s


Worst accuracy was about $\frac{1}{100}$. We can see that the runtime does vary. This is expected since the points are randomly positioned. 

- **What is the serial time? This will be your speedup denominator in E3.**

It was $T_1 = 1.085$ s.

****Done?** Note your serial time → commit**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 403837fbd30986107ca8c81c930cc4d4771e19c1 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Thu Mar 5 20:06:40 2026 +0000

    l04: exercise 1 done
```

### ****Exercises 2:** MC π Parallel**

**Parallelize with `Pool.map(); sweep 1. . . cpu count()` workers**

****Parallelize using `Pool.map` — each worker returns a hit count (an integer):****
- **Worker function estimate pi chunk(num samples): runs the loop, returns inside circle**
- **Main process: sum worker results, divide by total samples to get π̂**
- **Why good IPC? Input: one integer per task. Output: one integer per task. No arrays.**

****Sweep** `num_processes` from 1 to `os.cpu count()`. For each worker count: measure elapsed time and compute speedup.**

In [ ]:
# From slide 40

from multiprocessing import Pool
import os
import random
import time
import statistics


def estimate_pi_chunk(num_samples):
    inside_circle = 0
    for _ in range(num_samples):
        x, y = random.random(), random.random()
        if x*x + y*y <= 1:
            inside_circle += 1
    return inside_circle


def estimate_pi_parallel(num_samples, num_processes=4):
    samples_per_process = num_samples // num_processes
    tasks = [samples_per_process] * num_processes
    with Pool(processes=num_processes) as pool:
        results = pool.map(estimate_pi_chunk, tasks)
    return 4 * sum(results) / num_samples


if __name__ == '__main__':
    num_samples = 10_000_000
    for num_proc in range(1, os.cpu_count() + 1):
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            pi_est = estimate_pi_parallel(num_samples, num_proc)
            times.append(time.perf_counter() - t0)
        t = statistics.median(times)
        print(f"{num_proc:2d} workers: {t:.3f}s pi={pi_est:.6f}")

**Questions:**

- **Do all worker counts give the same $\hat{\pi}$? Why or why not?**

- **At which count do you first see a meaningful speedup?**

****Done?** Check $\hat{\pi}$ is consistent across runs → continue to E3**

### ****Exercises 3:** MC π Analysis**

**Speedup curve; efficiency table; back-solve implied serial fraction**

****Parallel efficiency:** $E_p = \frac{S_p}{p}$ — fraction of ideal speedup per core (100% = perfect). **For each worker count (1 to `cpu count()`), tabulate:** workers | time ($s$) | speedup $S_p$ | efficiency $E_p$(%)**

****Questions to discuss:****

**1. At which worker count $p^∗$ is speedup maximum?**

**2. Does speedup plateau or drop beyond $p^∗$? Why?**

**3. **Back-solve implied serial fraction:** $s = \frac{1/S_{p^*} - 1/p^*}{1 - 1/p^*}$ — what fraction of time is effectively serial (IPC overhead + spawning)?**

**4. **Mac M1/M2/M3 users:** do you see a slope change near worker 8 (E-cores)?**

****(Optional) Plot:** workers (x) vs. speedup (y) — actual and ideal linear.**

****Done?** Discuss with a neighbour → move on to MP2 milestones**

### ****Milestone 1:** Parallel Mandelbrot**

**Refactor to chunk-based Numba kernels**

In [ ]:
# From slide 45 (The updated once)

# mandelbrot_parallel.py (Tasks 1-3 are one continuous script)
import numpy as np
from numba import njit
from multiprocessing import Pool
import time, os, statistics, matplotlib.pyplot as plt
from pathlib import Path

@njit
def mandelbrot_pixel(c_real, c_imag, max_iter):
    z_real = z_imag = 0.0
    for i in range(max_iter):
        zr2 = z_real*z_real
        zi2 = z_imag*z_imag
        if zr2 + zi2 > 4.0: return i
        z_imag = 2.0*z_real*z_imag + c_imag
        z_real = zr2 - zi2 + c_real
    return max_iter

@njit
def mandelbrot_chunk(row_start, row_end, N,
                     x_min, x_max, y_min, y_max, max_iter):
    out = np.empty((row_end - row_start, N), dtype=np.int32)
    dx = (x_max - x_min) / N
    dy = (y_max - y_min) / N
    for r in range(row_end - row_start):
        c_imag = y_min + (r + row_start) * dy
        for col in range(N):
            out[r, col] = mandelbrot_pixel(x_min + col*dx, c_imag, max_iter)
    return out

def mandelbrot_serial(N, x_min, x_max, y_min, y_max, max_iter=100):
    return mandelbrot_chunk(0, N, N, x_min, x_max, y_min, y_max, max_iter)

**Refactor your MP1 Numba code into three functions (two `@njit`, one plain wrapper):**

**`mandelbrot pixel(c real, c imag, max iter)` The scalar kernel you already have from L03. Returns the escape iteration count for a single complex point.**

**`mandelbrot chunk(row start, row end, N, x min, x max, y min, y max, max iter)` Loops over rows [row start, row end) and all columns. Computes pixel coordinates from index + bounds — no arrays received as input. Returns a (row end - row start)×N int32 array.**

**`mandelbrot serial(N, x min, ...) Thin wrapper: calls mandelbrot chunk(0, N, ...)` — the whole grid as one chunk.**

****Done?** Serial result matches your L03 output → commit**

### ****Milestone 2:** Parallel Mandelbrot**

**Implement Pool.map wrapper; verify result**

In [ ]:
# From slide 47

# --- MP2 M2: add below M1 in mandelbrot_parallel.py ---
def _worker(args):
    return mandelbrot_chunk(*args)

def mandelbrot_parallel(N, x_min, x_max, y_min, y_max,
                        max_iter=100, n_workers=4):
    chunk_size = max(1, N // n_workers)
    chunks, row = [], 0
    while row < N:
        row_end = min(row + chunk_size, N)
        chunks.append((row, row_end, N, x_min, x_max, y_min, y_max, max_iter))
        row = row_end

    with Pool(processes=n_workers) as pool:
        pool.map(_worker, chunks)       # un-timed warm-up: Numba JIT in workers
        parts = pool.map(_worker, chunks)
    return np.vstack(parts)

if __name__ == '__main__':
    result = mandelbrot_parallel(1024, -2.5, 1.0, -1.25, 1.25, n_workers=4)

- **Write `mandelbrot_parallel(N, x min, x max, y min, y max, max iter, n workers)`**
    - **Build a list of chunk tuples: (row start, row end, N, x min, x max, y min, y max, max_iter)**
    - **Use pool.map( worker, chunks) to distribute across workers**
    - **Reassemble with np.vstack(parts)**

- **Pass only parameters — not a pre-computed grid**
    - **In MP1 we pre-created the full complex grid `C` and passed it to the iteration algorithm — passing a slice of that array to each worker would mean sending ∼8 KB per task over IPC**
    - **Instead: pass only scalar bounds `(x_min, x_max, y_min, y_max, N)` and let each worker compute its own coordinates**
    - **IPC input per task: ∼60 bytes (8 scalars) vs. ∼8 KB (1024-element float64 array)**
    - **IPC output per task is unavoidable: chunk size × N × 4 bytes (the result array)**

- **Wrapper function worker(args)**
    - **`pool.map` calls func(item) — it takes only one argument**
    - **mandelbrot chunk takes 8 arguments, so worker receives the packed tuple and unpacks it**
    - **Normally, this can be done with a lambda function, but lambdas are not picklable, so it must be defined as function at module level.**

****Done?** Verify result matches serial → commit**

### ****Milestone 3:** Parallel Mandelbrot**

**Benchmark sweep; speedup and efficiency table**

****Serial baseline:** call mandelbrot serial 3 times after your M1 warm-up (Numba already compiled in main process); take median**

- **Where to place the timer:**
    - **Start `t0` immediately before `pool.map()`**
    - **Stop immediately after `np.vstack(parts)` — include assembly time**
    - **Keep Pool creation *outside* the timed region**

In [ ]:
# From slide 49

# --- MP2 M3: benchmark (in __main__ block) ---
N, max_iter = 1024, 100
X_MIN, X_MAX, Y_MIN, Y_MAX = -2.5, 1.0, -1.25, 1.25

# Serial baseline (Numba already warm after M1 warm-up)
times = []
for _ in range(3):
    t0 = time.perf_counter()
    mandelbrot_serial(N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter)
    times.append(time.perf_counter() - t0)
t_serial = statistics.median(times)

for n_workers in range(1, os.cpu_count() + 1):
    chunk_size = max(1, N // n_workers)
    chunks, row = [], 0
    while row < N:
        end = min(row + chunk_size, N)
        chunks.append((row, end, N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter))
        row = end
    with Pool(processes=n_workers) as pool:
        pool.map(_worker, chunks)           # warm-up: Numba JIT in all workers
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            np.vstack(pool.map(_worker, chunks))
            times.append(time.perf_counter() - t0)
    t_par = statistics.median(times)
    speedup = t_serial / t_par
    print(f"{n_workers:2d} workers: {t_par:.3f}s, speedup={speedup:.2f}x, eff={speedup/n_workers*100:.0f}%")

**Sweep n workers from 1 to os.cpu count(); for each worker count:**

**1. Create a fresh `Pool(processes=n_workers)`**

**2. Run one **un-timed** `pool.map(_worker, chunks)` — this triggers Numba JIT compilation in every worker process; without this, JIT time is included in your measurement**

**3. Run 3 **timed** `pool.map` calls; take `statistics.median()`**

**4. Record: time, speedup $S_p = t_{serial}/t_p$, efficiency $E_p = S_p/p$**

****Done?** Record speedup table in performance notebook (MP2) → commit**